In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 07:33:43.517489: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 07:33:44.214720: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 07:33:46,019 [DEBUG] [LogService] Rain is initialized
2023-07-02 07:33:46,019 [DEBUG] [LogService] Creating coordinator
2023-07-02 07:33:46,019 [DEBUG] [LogService] Coordinator is initialized
2023-07-02 07:33:46,020 [DEBUG] [LogService] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_sync()

2023-07-02 07:33:46,026 [DEBUG] [LogService] Creating workers
2023-07-02 07:33:46,032 [INFO] [LogService] provisioner is serving
2023-07-02 07:33:46,033 [DEBUG] [LogService] Starting coordinator
2023-07-02 07:33:46,034 [INFO] [LogService] coordinator is serving
2023-07-02 07:33:46,035 [DEBUG] [LogService] sending the num of workers to the provisioner
2023-07-02 07:33:46,039 [DEBUG] [LogService] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 07:33:46,040 [DEBUG] [LogService] sent Success receiving the number of workers to the provisioner
2023-07-02 07:33:46,041 [DEBUG] [LogService] Creating 3 workers
2023-07-02 07:33:46,042 [INFO] [LogService] Worker is running on port: 50151
2023-07-02 07:33:46,044 [INFO] [LogService] Worker is running on port: 50152
2023-07-02 07:33:46,045 [INFO] [LogService] Worker is running on port: 50153
2023-07-02 07:33:46,046 [DEBUG] [LogService] [Created workers]
 IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports:

Epoch 1/2
Epoch 1/2


2023-07-02 07:34:21.393077: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.


Epoch 1/2
157/157 [==============================] - 1s 6ms/step - loss: 0.6956 - accuracy: 0.7808
Epoch 2/2
157/157 [==============================] - 2s 9ms/step - loss: 0.7016 - accuracy: 0.7786
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.6987 - accuracy: 0.7785
Epoch 2/2
157/157 [==============================] - 1s 7ms/step - loss: 0.3086 - accuracy: 0.9112
sending data to coordinator


2023-07-02 07:34:25,374 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:25,375 [INFO] [LogService] thread 1 is done
2023-07-02 07:34:25,465 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_1_trained.pkl in coordinator
2023-07-02 07:34:25,523 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:25,543 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:25,544 [INFO] [LogService] thread 2 is done
2023-07-02 07:34:25,623 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_1_trained.pkl in coordinator
2023-07-02 07:34:25,623 [INFO] [LogService] thread 3 is done
2023-07-02 07:34:25,708 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_1_trained.pkl in coordinator
2023-07-02 07:34:25,788 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:34:25,863 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:3

...

2023-07-02 07:34:46,539 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:46,541 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:46,542 [INFO] [LogService] thread 1 is done
2023-07-02 07:34:46,544 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:34:46,621 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
2023-07-02 07:34:46,622 [INFO] [LogService] thread 2 is done
2023-07-02 07:34:46,700 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
2023-07-02 07:34:46,700 [INFO] [LogService] thread 3 is done
2023-07-02 07:34:46,777 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
2023-07-02 07:34:46,855 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:34:46,941 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:3

Epoch 1/2
Epoch 1/2
Epoch 1/2


2023-07-02 07:35:06.172233: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 62720000 exceeds 10% of free system memory.


157/157 [==============================] - 2s 6ms/step - loss: 0.1776 - accuracy: 0.9457
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1915 - accuracy: 0.9443
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1826 - accuracy: 0.9460
Epoch 2/2
157/157 [==============================] - 1s 7ms/step - loss: 0.1555 - accuracy: 0.9528
sending data to coordinator


2023-07-02 07:35:09,944 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:35:10,115 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:35:10,127 [DEBUG] [LogService] coordinator received: Executed! from worker
2023-07-02 07:35:10,128 [INFO] [LogService] thread 1 is done
2023-07-02 07:35:10,204 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
2023-07-02 07:35:10,204 [INFO] [LogService] thread 2 is done
2023-07-02 07:35:10,280 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
2023-07-02 07:35:10,281 [INFO] [LogService] thread 3 is done
2023-07-02 07:35:10,361 [DEBUG] [LogService] Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
2023-07-02 07:35:10,436 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:35:10,509 [DEBUG] [LogService] coordinator received: Success! from divider
2023-07-02 07:3

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0945 - accuracy: 0.9697

Test accuracy: 97.0%
